# Melanoma & Nevus Dermoscopy Dataset - Metadata Preprocessing

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/metadata.csv")
print(df.shape)
df.head()

(18133, 32)


,isic_id,attribution,copyright_license,acquisition_day,age_approx,anatom_site_general,anatom_site_special,benign_malignant,clin_size_long_diam_mm,concomitant_biopsy,...,mel_class,mel_mitotic_index,mel_thick_mm,mel_type,mel_ulcer,melanocytic,nevus_type,patient_id,personal_hx_mm,sex
0,ISIC_0000002,Anonymous,CC-0,NaN,60.0,upper extremity,NaN,malignant,NaN,True,...,NaN,NaN,NaN,NaN,NaN,True,NaN,NaN,NaN,female
1,ISIC_0000004,Anonymous,CC-0,NaN,80.0,posterior torso,NaN,malignant,NaN,True,...,NaN,NaN,NaN,NaN,NaN,True,NaN,NaN,NaN,male
2,ISIC_0000013,Anonymous,CC-0,NaN,30.0,posterior torso,NaN,malignant,NaN,True,...,NaN,NaN,NaN,NaN,NaN,True,NaN,NaN,NaN,female
3,ISIC_0000022,Anonymous,CC-0,NaN,55.0,lower extremity,NaN,malignant,NaN,True,...,NaN,NaN,NaN,NaN,NaN,True,NaN,NaN,NaN,female
4,ISIC_0000026,Anonymous,CC-0,NaN,30.0,lower extremity,NaN,malignant,NaN,True,...,NaN,NaN,NaN,NaN,NaN,True,NaN,NaN,NaN,female


##### 1. Khám phá dữ liệu (EDA)

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18133 entries, 0 to 18132
Data columns (total 32 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   isic_id                 18133 non-null  object 
 1   attribution             18133 non-null  object 
 2   copyright_license       18133 non-null  object 
 3   acquisition_day         1422 non-null   float64
 4   age_approx              17675 non-null  float64
 5   anatom_site_general     16998 non-null  object 
 6   anatom_site_special     750 non-null    object 
 7   benign_malignant        18126 non-null  object 
 8   clin_size_long_diam_mm  2537 non-null   float64
 9   concomitant_biopsy      18133 non-null  bool   
 10  dermoscopic_type        4292 non-null   object 
 11  diagnosis               18133 non-null  object 
 12  diagnosis_1             18133 non-null  object 
 13  diagnosis_2             18133 non-null  object 
 14  diagnosis_3             18107 non-null

In [3]:
# % giá trị thiếu theo từng cột
df.isna().mean().sort_values(ascending=False) * 100

mel_mitotic_index         99.696686
mel_type                  98.908068
mel_ulcer                 98.803287
mel_thick_mm              96.260961
anatom_site_special       95.863895
fitzpatrick_skin_type     95.130425
mel_class                 94.430045
nevus_type                92.577069
diagnosis_5               92.428170
acquisition_day           92.157944
clin_size_long_diam_mm    86.008934
family_hx_mm              83.064027
personal_hx_mm            82.275409
diagnosis_4               82.148569
dermoscopic_type          76.330447
patient_id                72.448023
lesion_id                 22.533502
anatom_site_general        6.259306
age_approx                 2.525782
melanocytic                2.520267
sex                        2.189378
diagnosis_3                0.143385
benign_malignant           0.038604
image_type                 0.000000
diagnosis_confirm_type     0.000000
attribution                0.000000
diagnosis_2                0.000000
diagnosis_1                0

In [4]:
# Kiểm tra các cột zero-variance / near-zero-variance (gần như hằng số, không có khả năng phân biệt)
check_constant_cols = ['diagnosis_confirm_type', 'image_type', 'concomitant_biopsy', 'melanocytic']

for c in check_constant_cols:
    print(f"--- {c} ---")
    print((df[c].value_counts(dropna=False, normalize=True) * 100).round(2))
    print()

--- diagnosis_confirm_type ---
diagnosis_confirm_type
histopathology    100.0
Name: proportion, dtype: float64

--- image_type ---
image_type
dermoscopic    100.0
Name: proportion, dtype: float64

--- concomitant_biopsy ---
concomitant_biopsy
True     99.71
False     0.29
Name: proportion, dtype: float64

--- melanocytic ---
melanocytic
True    97.48
NaN      2.52
Name: proportion, dtype: float64



##### 2. Loại bỏ các cột không cần thiết cho việc xây dựng model

Gộp 4 nhóm cột không nên đưa vào model, loại bỏ trong một bước duy nhất:

- **Hằng số (constant)**: `diagnosis_confirm_type`, `image_type` — 100% cùng 1 giá trị, không mang tín hiệu phân biệt.
- **Hành chính (administrative)**: `copyright_license`, `acquisition_day`, `dermoscopic_type` — thông tin pháp lý/thiết bị chụp, không liên quan chẩn đoán. (`attribution` được giữ lại — không đưa vào model như feature, nhưng cần thiết để phân tích missingness theo nguồn dữ liệu ở bước EDA.)
- **Rò rỉ nhãn (target-related / label leakage)**: `benign_malignant`, `diagnosis_1`–`diagnosis_5`, `mel_class`, `mel_type`, `mel_thick_mm`, `mel_ulcer`, `mel_mitotic_index`, `nevus_type` — trùng hoặc suy ra trực tiếp từ nhãn `diagnosis`.
- **Quá thưa (sparse, >80% missing)**: `anatom_site_special`, `clin_size_long_diam_mm`, `family_hx_mm`, `fitzpatrick_skin_type`, `personal_hx_mm` — không đủ dữ liệu để làm feature đáng tin cậy.

Giữ lại: khóa (`isic_id`, `lesion_id`, `patient_id`), nhãn (`diagnosis`), feature có missing thấp + ý nghĩa lâm sàng độc lập với nhãn (`age_approx`, `sex`, `anatom_site_general`), và `attribution` (phục vụ phân tích nguồn dữ liệu).

In [5]:
constant_cols = ['diagnosis_confirm_type', 'image_type']

administrative_cols = ['copyright_license', 'acquisition_day', 'dermoscopic_type']

target_related_cols = [
    'benign_malignant', 'diagnosis_1', 'diagnosis_2', 'diagnosis_3', 'diagnosis_4', 'diagnosis_5',
    'mel_class', 'mel_type', 'mel_thick_mm', 'mel_ulcer', 'mel_mitotic_index', 'nevus_type',
]

sparse_cols = ['anatom_site_special', 'clin_size_long_diam_mm', 'family_hx_mm', 'fitzpatrick_skin_type', 'personal_hx_mm']

cols_to_drop = constant_cols + administrative_cols + target_related_cols + sparse_cols

df_clean = df.drop(columns=cols_to_drop)
print(f"Đã drop {len(cols_to_drop)} cột, còn lại {df_clean.shape[1]} cột")
df_clean.info()

Đã drop 22 cột, còn lại 10 cột
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18133 entries, 0 to 18132
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   isic_id              18133 non-null  object 
 1   attribution          18133 non-null  object 
 2   age_approx           17675 non-null  float64
 3   anatom_site_general  16998 non-null  object 
 4   concomitant_biopsy   18133 non-null  bool   
 5   diagnosis            18133 non-null  object 
 6   lesion_id            14047 non-null  object 
 7   melanocytic          17676 non-null  object 
 8   patient_id           4996 non-null   object 
 9   sex                  17736 non-null  object 
dtypes: bool(1), float64(1), object(8)
memory usage: 1.3+ MB


##### 3. Định nghĩa vai trò cột

Phân loại 9 cột còn lại trong `df_clean` theo vai trò sử dụng:

- **Khóa (key)**: `isic_id`, `lesion_id`, `patient_id` — dùng để nối với ảnh và group-split train/val/test, không đưa vào model như một feature.
- **Nhãn (label)**: `diagnosis` — biến mục tiêu cần dự đoán (melanoma/nevus).
- **Feature khuyến nghị**: `age_approx`, `sex`, `anatom_site_general` — missing thấp, có ý nghĩa lâm sàng độc lập với nhãn.
- **Giữ lại nhưng không khuyến nghị làm feature**: `concomitant_biopsy`, `melanocytic` — gần như hằng số (near-zero variance, lần lượt 99.7% và 97.5% cùng 1 giá trị) nên ít giá trị phân biệt, vẫn giữ trong `df_clean` để tham khảo/phân tích phụ thay vì xóa hẳn.

In [6]:
key_cols = ['isic_id', 'lesion_id', 'patient_id']          # khóa: nối ảnh & group-split, KHÔNG đưa vào model
label_col = 'diagnosis'                                     # nhãn (melanoma / nevus)
feature_cols = ['age_approx', 'sex', 'anatom_site_general']  # feature khuyến nghị cho model
near_constant_keep_cols = ['concomitant_biopsy', 'melanocytic']  # giữ lại, không khuyến nghị làm feature

print("Key columns             :", key_cols)
print("Label column             :", label_col)
print("Feature columns          :", feature_cols)
print("Giữ lại (không làm feature):", near_constant_keep_cols)

df_clean[key_cols + [label_col] + feature_cols + near_constant_keep_cols].head()

Key columns             : ['isic_id', 'lesion_id', 'patient_id']
Label column             : diagnosis
Feature columns          : ['age_approx', 'sex', 'anatom_site_general']
Giữ lại (không làm feature): ['concomitant_biopsy', 'melanocytic']


,isic_id,lesion_id,patient_id,diagnosis,age_approx,sex,anatom_site_general,concomitant_biopsy,melanocytic
0,ISIC_0000002,NaN,NaN,melanoma,60.0,female,upper extremity,True,True
1,ISIC_0000004,NaN,NaN,melanoma,80.0,male,posterior torso,True,True
2,ISIC_0000013,NaN,NaN,melanoma,30.0,female,posterior torso,True,True
3,ISIC_0000022,NaN,NaN,melanoma,55.0,female,lower extremity,True,True
4,ISIC_0000026,NaN,NaN,melanoma,30.0,female,lower extremity,True,True


##### 4. Chia train / validation / test theo nhóm (group-split)

Nhiều ảnh thuộc cùng 1 `lesion_id` (trung bình ~1.91 ảnh/lesion, có lesion tới 31 ảnh) nên **không được split ngẫu nhiên theo từng ảnh** — phải group theo `lesion_id` (ưu tiên), fallback `patient_id`, fallback chính `isic_id` nếu thiếu cả hai. Việc fill-in missing values sẽ làm **sau** bước này, dùng thống kê tính riêng trên tập train để tránh leakage.

In [7]:
from sklearn.model_selection import GroupShuffleSplit

# Group key: ưu tiên lesion_id, fallback patient_id, fallback chính isic_id (nếu thiếu cả hai)
group_key = df_clean['lesion_id'].fillna(df_clean['patient_id']).fillna(df_clean['isic_id'])

# Bước 1: tách test (15%)
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_idx, test_idx = next(gss1.split(df_clean, groups=group_key))

df_train_val = df_clean.iloc[train_val_idx]
df_test = df_clean.iloc[test_idx]

# Bước 2: tách train/val từ phần còn lại (~17.65% của train_val => ~15% của tổng)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)
train_idx, val_idx = next(gss2.split(df_train_val, groups=group_key.iloc[train_val_idx]))

df_train = df_train_val.iloc[train_idx]
df_val = df_train_val.iloc[val_idx]

print(f"Train: {len(df_train)} ({len(df_train)/len(df_clean):.1%})")
print(f"Val  : {len(df_val)} ({len(df_val)/len(df_clean):.1%})")
print(f"Test : {len(df_test)} ({len(df_test)/len(df_clean):.1%})")

# Kiểm tra không có group nào bị chia sẻ giữa các tập
g_train = set(group_key.iloc[train_val_idx].iloc[train_idx])
g_val = set(group_key.iloc[train_val_idx].iloc[val_idx])
g_test = set(group_key.iloc[test_idx])
print("Overlap train/val :", len(g_train & g_val))
print("Overlap train/test:", len(g_train & g_test))
print("Overlap val/test  :", len(g_val & g_test))

Train: 12696 (70.0%)
Val  : 2752 (15.2%)
Test : 2685 (14.8%)
Overlap train/val : 0
Overlap train/test: 0
Overlap val/test  : 0


##### 5. Lưu các tập đã chia

Lưu `df_train`, `df_val`, `df_test` vào `data/splitted/` để dùng cho các bước sau (fill missing, training model...). Chưa fill-in missing values ở bước này — việc đó cần tính thống kê riêng trên `df_train` trước khi áp dụng xuống cả 3 tập.

In [8]:
df_train.to_csv("../data/splitted/train.csv", index=False)
df_val.to_csv("../data/splitted/val.csv", index=False)
df_test.to_csv("../data/splitted/test.csv", index=False)

print("Đã lưu: ../data/splitted/train.csv, val.csv, test.csv")

Đã lưu: data/splitted/train.csv, val.csv, test.csv
